# PulmoAI: Exploratory Analysis and Inference Walkthrough

This notebook demonstrates how to interact with the **PulmoAI** framework:
1. Loading the centralized configuration
2. Inspecting the IQ-OTH/NCCD dataset
3. Instantiating the ResNet50 + CBAM Attention model
4. Running inference with Monte Carlo Dropout uncertainty estimation
5. Generating Explainable AI (Grad-CAM) visual attention heatmaps

In [ ]:
import os
import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt
import numpy as np

# Ensure project root is in path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

from src.utils.config import load_config
from src.models.resnet50 import create_model
from src.explainability.engine import ExplainabilityEngine
from src.robustness.uncertainty import UncertaintyQuantifier
from src.robustness.input_validation import InputValidator

cfg = load_config(str(ROOT_DIR / 'configs' / 'config.yaml'))
print(f"Project: {cfg['project']['name']} v{cfg['project']['version']}")
print(f"Classes: {cfg['data']['class_names']}")

## 1. Load Pretrained ResNet50-CBAM Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Execution Device: {device}")

model = create_model(num_classes=cfg['model']['num_classes'], pretrained=False)
checkpoint_path = ROOT_DIR / cfg['checkpoints']['dir'] / cfg['checkpoints']['best_model_filename']

if checkpoint_path.exists():
    state = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(state)
    print(f"Loaded trained checkpoint from: {checkpoint_path}")
else:
    print(f"Checkpoint not found at: {checkpoint_path}")

model = model.to(device)
model.eval()

## 2. Initialize XAI and Uncertainty Engines

In [ ]:
xai_engine = ExplainabilityEngine(
    model=model,
    device=device,
    class_names=cfg['data']['class_names'],
    mean=cfg['data']['normalization']['mean'],
    std=cfg['data']['normalization']['std']
)

uq_quantifier = UncertaintyQuantifier(
    model=model,
    device=device,
    class_names=cfg['data']['class_names'],
    uncertainty_threshold=cfg['robustness']['uncertainty']['uncertainty_threshold'],
    confidence_threshold=cfg['robustness']['uncertainty']['confidence_threshold']
)
print("XAI and Uncertainty engines successfully initialized.")

## 3. Run Inference on a Sample CT Scan

In [ ]:
sample_path = ROOT_DIR / 'static' / 'samples' / 'malignant_sample.jpg'
if not sample_path.exists():
    # Fallback to test directory
    test_files = list((ROOT_DIR / 'dataset' / 'test').glob('*/*.*'))
    sample_path = test_files[0] if test_files else None

if sample_path:
    print(f"Analyzing sample: {sample_path.name}")
    rgb_uint8, rgb_float, tensor = xai_engine.load_image(str(sample_path))
    
    # Compute Uncertainty via Monte Carlo Dropout
    uq_res = uq_quantifier.estimate(tensor, num_samples=20)
    print(f"Prediction:    {uq_res['predicted_class']}")
    print(f"Confidence:    {uq_res['confidence']:.2%}")
    print(f"Uncertainty:   {uq_res['uncertainty']:.4f}")
    print(f"Entropy:       {uq_res['entropy']:.4f}")
    print(f"Reliability:   {'RELIABLE' if uq_res['is_reliable'] else 'AMBIGUOUS'}")
    
    # Generate Grad-CAM
    gcam = xai_engine.gradcam.explain(tensor, rgb_float, target_class=uq_res['predicted_class_idx'])
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(rgb_uint8)
    axes[0].set_title(f"Input CT Slice\nPred: {uq_res['predicted_class']} ({uq_res['confidence']:.1%})")
    axes[0].axis('off')
    
    axes[1].imshow(gcam['overlay'])
    axes[1].set_title("Grad-CAM Attribution Heatmap")
    axes[1].axis('off')
    plt.show()
else:
    print("No sample image found. Place an image in static/samples/ or dataset/test/ to test.")